In [1]:
"""
===============================================================================
Script Name: 01_gpu_ode_calibration.py
Description: Loads chunked tensor segments and trains a Physics-Informed 
             ODE to calibrate thermal parameters. Implements a dynamic
             state-dependent cooling proxy h(T) to simulate fan curves.
             Includes interactive GPU selection, auto-resume, and CSV logging.
             Optimized with background double-buffered I/O and manual batching.
===============================================================================
"""

import os
import sys
import time
import warnings
import json
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from pathlib import Path
from tqdm import tqdm
from datetime import timedelta
import gc
import threading

# --- 1. CONFIGURATION ---
warnings.filterwarnings('ignore')

# Hyperparameters
BATCH_SIZE = 25000
EPOCHS = 50
LEARNING_RATE = 0.01
DT = 0.11

# --- 2. GPU SELECTION & INITIALIZATION ---
def select_gpu():
    """
    Selects GPU via CLI arg, env var, or interactive prompt (in that order).
    """
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            gpu_id = int(sys.argv[i + 1])
            if gpu_id < torch.cuda.device_count():
                print(f"[SYSTEM] GPU {gpu_id} selected via CLI argument.")
                return gpu_id
            else:
                print(f"[!] CLI GPU {gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        gpu_id = int(env_gpu)
        if gpu_id < torch.cuda.device_count():
            print(f"[SYSTEM] GPU {gpu_id} selected via GPU_ID environment variable.")
            return gpu_id
        else:
            print(f"[!] Env GPU_ID={gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    print("\n[SYSTEM] Live GPU Status:")
    try:
        subprocess.run(["nvidia-smi"])
    except FileNotFoundError:
        print("[WARNING] nvidia-smi not found. Cannot display GPU stats.")

    if not torch.cuda.is_available():
        print("[WARNING] No CUDA GPUs detected. Falling back to CPU.")
        return None

    while True:
        gpu_id = input("\n[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']: ").strip()
        if not gpu_id:
            print("[SYSTEM] Defaulting to GPU 0.")
            return 0
        if gpu_id.isdigit():
            gpu_id_int = int(gpu_id)
            if gpu_id_int < torch.cuda.device_count():
                print(f"[SYSTEM] Manually locking script to GPU {gpu_id_int}.")
                return gpu_id_int
            else:
                print(f"[!] GPU {gpu_id_int} not found. Available GPUs: 0-{torch.cuda.device_count()-1}")
                continue
        print("[!] Invalid input. Please enter a valid integer ID.")

# --- 3. TERMINAL LOGGING UTILITY ---
class DualLogger:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log = open(filepath, "a", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def close(self):
        self.log.close()

# --- 4. CHUNK PREFETCHER (DOUBLE-BUFFERED I/O) ---
class ChunkPrefetcher:
    """
    Loads the next .pt chunk from disk in a background thread while the GPU 
    processes the current chunk. Eliminates ~10-15s of GPU idle time per chunk 
    transition by overlapping disk I/O with GPU compute.
    """
    def __init__(self, chunk_files):
        self.chunk_files = chunk_files
        self._next_chunk = None
        self._thread = None

    def _load_in_background(self, filepath):
        """Target function for the background thread."""
        self._next_chunk = torch.load(filepath, map_location='cpu', weights_only=False)

    def prefetch(self, chunk_idx):
        """Start loading chunk at chunk_idx in the background (no-ops if out of bounds)."""
        if chunk_idx < len(self.chunk_files):
            self._thread = threading.Thread(
                target=self._load_in_background,
                args=(self.chunk_files[chunk_idx],)
            )
            self._thread.start()

    def get(self):
        """Block until the prefetched chunk is ready, then return it."""
        if self._thread is not None:
            self._thread.join()
            self._thread = None
        chunk = self._next_chunk
        self._next_chunk = None
        return chunk

# --- 4. PHYSICS MODEL (THE ODE) ---
class ThermalODEModel(nn.Module):
    def __init__(self):
        super(ThermalODEModel, self).__init__()
        
        # Define Priors (Starting Guesses)
        prior_C         = 141.735
        prior_k01       = 0.0780
        prior_k10       = 0.0281
        prior_q         = 0.0
        
        # Dynamic Fan Curve Priors
        prior_h_base    = 2.0   # Baseline passive cooling
        prior_h_active  = 8.0   # Max additional cooling from fans
        prior_T_thresh  = 65.0  # Temp where fans start to aggressively ramp up
        prior_beta      = 0.5   # Steepness of the fan curve ramp
        
        # Define Physical Bounds [lower, upper]
        self.bounds = {
            'C':          (100.0, 1000.0),
            'k01':        (0.0, 0.5),
            'k10':        (0.0, 0.5),
            'q0':         (-10.0, 10.0),
            'q1':         (-10.0, 10.0),
            'h_base_0':   (0.5, 10.0),
            'h_base_1':   (0.5, 10.0),
            'h_active_0': (0.0, 30.0),
            'h_active_1': (0.0, 30.0),
            'T_thresh_0': (40.0, 85.0),
            'T_thresh_1': (40.0, 85.0),
            'beta_0':     (0.05, 2.0),
            'beta_1':     (0.05, 2.0)
        }
        
        # Initialize Raw Parameters (Inverse Sigmoid)
        def inv_sigmoid(val, low, high):
            norm = (val - low) / (high - low)
            norm = max(min(norm, 0.999), 0.001)
            return torch.log(torch.tensor(norm / (1.0 - norm)))

        # General Params
        self.raw_C          = nn.Parameter(inv_sigmoid(prior_C, *self.bounds['C']))
        self.raw_k01        = nn.Parameter(inv_sigmoid(prior_k01, *self.bounds['k01']))
        self.raw_k10        = nn.Parameter(inv_sigmoid(prior_k10, *self.bounds['k10']))
        self.raw_q0         = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q0']))
        self.raw_q1         = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q1']))
        
        # GPU 0 Fan Curve Params
        self.raw_h_base_0   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_0']))
        self.raw_h_active_0 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_0']))
        self.raw_T_thresh_0 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_0']))
        self.raw_beta_0     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_0']))
        
        # GPU 1 Fan Curve Params
        self.raw_h_base_1   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_1']))
        self.raw_h_active_1 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_1']))
        self.raw_T_thresh_1 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_1']))
        self.raw_beta_1     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_1']))

    def get_physical_params(self):
        """Applies sigmoid to raw parameters to bind them within physical reality."""
        p = {}
        for name, bound in self.bounds.items():
            raw_val = getattr(self, f"raw_{name}")
            low, high = bound
            p[name] = low + torch.sigmoid(raw_val) * (high - low)
        return p

    def forward(self, P0, P1, T0_init, T1_init, T_amb):
        """Simulates the thermal trajectories using Forward Euler integration."""
        params = self.get_physical_params()
        
        batch_size, seq_len = P0.shape
        
        # Pre-allocate to prevent torch.stack memory fragmentation
        T0_preds = torch.empty((batch_size, seq_len), device=P0.device)
        T1_preds = torch.empty((batch_size, seq_len), device=P1.device)
        
        T0_curr = T0_init
        T1_curr = T1_init
        T0_preds[:, 0] = T0_curr
        T1_preds[:, 0] = T1_curr
        
        inv_C = 1.0 / params['C']
        
        # Euler Integration Loop
        for t in range(seq_len - 1):
            # 1. Calculate Dynamic Cooling Rate h(T) for current timestep
            h0_curr = params['h_base_0'] + params['h_active_0'] * torch.sigmoid(params['beta_0'] * (T0_curr - params['T_thresh_0']))
            h1_curr = params['h_base_1'] + params['h_active_1'] * torch.sigmoid(params['beta_1'] * (T1_curr - params['T_thresh_1']))
            
            # 2. Calculate differentials
            dT0 = (P0[:, t] + params['k01'] * P1[:, t] - h0_curr * T0_curr + h0_curr * T_amb + params['q0']) * inv_C
            dT1 = (P1[:, t] + params['k10'] * P0[:, t] - h1_curr * T1_curr + h1_curr * T_amb + params['q1']) * inv_C
            
            # 3. Euler Step
            T0_curr = T0_curr + DT * dT0
            T1_curr = T1_curr + DT * dT1
            
            T0_preds[:, t+1] = T0_curr
            T1_preds[:, t+1] = T1_curr
            
        return T0_preds, T1_preds


# --- 5. MAIN EXECUTION ---
def main():
    global script_name
    try:
        script_path = Path(__file__).resolve()
        script_name = script_path.stem
        project_root = script_path.parent.parent.parent
    except NameError:
        script_name = "01_gpu_ode_calibration"
        current_dir = Path.cwd() 
        project_root = current_dir.parent.parent
    
    # Device Setup
    gpu_id = select_gpu()
    if gpu_id is not None:
        torch.cuda.set_device(gpu_id)
        DEVICE = torch.device(f"cuda:{gpu_id}")
    else:
        DEVICE = torch.device("cpu")

    # Versioning & Path Initialization
    user_prefix = input("\nEnter version prefix (e.g., v1_base) [Press Enter for default 'v1']: ").strip()
    VERSION_PREFIX = user_prefix if user_prefix else "v1"

    # Input Data Directory
    data_dir = project_root / "Implementation" / "data" / "mit-supercloud-dataset" / "gpu" / "dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_tensors"
    if not data_dir.exists():
        print(f"[ERROR] Cannot find tensor directory: {data_dir}")
        return

    # Output Save Directory
    prefix = "/home/sanke24839/"
    SAVE_DIR = project_root / "Implementation" / "models" / VERSION_PREFIX
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    
    log_path = SAVE_DIR / f"{script_name}_terminal_output.txt"
    sys.stdout = DualLogger(log_path)
    
    start_time = time.perf_counter()
    print("=" * 70)
    print(f"--- STARTING: {script_name.upper()} ---")
    print(f"[*] Version Prefix: {VERSION_PREFIX}")
    print(f"[*] Device: {DEVICE}")
    print(f"[*] Saving to: {str(SAVE_DIR).replace(prefix, '')}")
    print("=" * 70)

    # Background GPU Time-Series Logging
    gpu_logger_process = None
    if gpu_id is not None:
        gpu_csv_path = SAVE_DIR / f"{VERSION_PREFIX}_gpu{gpu_id}_timeseries.csv"
        try:
            # Queries specific metrics for the chosen GPU (-i), loops every 5 seconds (-l 5), writes to file (-f)
            smi_cmd = [
                "nvidia-smi",
                "--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,temperature.gpu,power.draw",
                "--format=csv",
                "-i", str(gpu_id),
                "-l", "5", 
                "-f", str(gpu_csv_path)
            ]
            gpu_logger_process = subprocess.Popen(smi_cmd)
            print(f"[*] Started continuous GPU logging to: {gpu_csv_path.name}")
        except FileNotFoundError:
            print("[WARNING] nvidia-smi not found. Continuous GPU logging disabled.")

    # Setup Model, Optimizer, & Criterions
    model = ThermalODEModel().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    mse_criterion = nn.MSELoss(reduction='none') # For optimization
    mae_criterion = nn.L1Loss(reduction='none')  # For interpretable logging

    # Auto-Resume Logic
    RESUME_PATH = SAVE_DIR / f"{VERSION_PREFIX}_resume_checkpoint.pt"
    BEST_PATH = SAVE_DIR / f"{VERSION_PREFIX}_best_model.pt"
    CSV_LOG_PATH = SAVE_DIR / "training_log.csv"

    start_epoch = 1
    best_val_loss = float('inf')

    if RESUME_PATH.exists():
        print(f"\n[SYSTEM] Found existing checkpoint at: {RESUME_PATH.name}")
        print(f"[SYSTEM] Resuming training...")
        
        checkpoint = torch.load(RESUME_PATH, map_location=DEVICE, weights_only=False)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        
        # Restore RNG states
        torch.set_rng_state(checkpoint['torch_rng'].cpu().to(torch.uint8) if isinstance(checkpoint['torch_rng'], torch.Tensor) else torch.ByteTensor(checkpoint['torch_rng']))
        np.random.set_state(checkpoint['numpy_rng'])
        
        if torch.cuda.is_available() and 'cuda_rng' in checkpoint and checkpoint['cuda_rng'] is not None:
            cuda_rng = checkpoint['cuda_rng']
            torch.cuda.set_rng_state(cuda_rng.cpu().to(torch.uint8) if isinstance(cuda_rng, torch.Tensor) else torch.ByteTensor(cuda_rng))
            
        print(f"[SYSTEM] Resumed from Epoch {checkpoint['epoch']}. Best Val RMSE: {best_val_loss:.6f}")
    else:
        print(f"\n[SYSTEM] No checkpoint found. Starting fresh training.")
        with open(CSV_LOG_PATH, "w") as f:
            f.write("epoch,train_rmse,val_rmse,train_mae,val_mae,current_lr,C,h_base_0,h_base_1,h_active_0,h_active_1,T_thresh_0,T_thresh_1,beta_0,beta_1,k01,k10,q0,q1,epoch_time_sec\n")

    # Discover chunks
    train_chunks = sorted(list(data_dir.glob("train_segments_part*.pt")))
    val_chunks = sorted(list(data_dir.glob("val_segments_part*.pt")))
    print(f"Found {len(train_chunks)} train chunks and {len(val_chunks)} val chunks.")

    # Pre-load validation data into RAM (loaded ONCE, reused every epoch)
    print(f"\n[SYSTEM] Pre-loading {len(val_chunks)} validation chunk(s) into RAM...")
    val_chunks_data = []
    for chunk_file in val_chunks:
        chunk_data = torch.load(chunk_file, map_location='cpu', weights_only=False)
        val_chunks_data.append(chunk_data)
    print(f"[SYSTEM] Validation data cached. ({len(val_chunks_data)} chunk(s) ready)")

    try:
        # Training Loop
        for epoch in range(start_epoch, EPOCHS + 1):
            epoch_start_time = time.perf_counter()
            print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
            
            # 1. Training Phase
            model.train()
            train_mse_accum = 0.0
            train_mae_accum = 0.0
            train_steps = 0
            
            prefetcher = ChunkPrefetcher(train_chunks)
            prefetcher.prefetch(0)
            
            for chunk_idx in range(len(train_chunks)):
                chunk_data = prefetcher.get()
                prefetcher.prefetch(chunk_idx + 1)
                
                total_size = chunk_data['P0'].shape[0]
                indices = torch.randperm(total_size)
                num_batches = int(np.ceil(total_size / BATCH_SIZE))
                
                for start_idx in tqdm(range(0, total_size, BATCH_SIZE), total=num_batches, desc=f"Train Chunk {chunk_idx+1}/{len(train_chunks)}", leave=False):
                    end_idx = min(start_idx + BATCH_SIZE, total_size)
                    batch_idx = indices[start_idx:end_idx]
                    
                    # Manual Batch Slicing & GPU Transfer
                    P0 = chunk_data['P0'][batch_idx].to(DEVICE, non_blocking=True)
                    P1 = chunk_data['P1'][batch_idx].to(DEVICE, non_blocking=True)
                    T0_true = chunk_data['T0'][batch_idx].to(DEVICE, non_blocking=True)
                    T1_true = chunk_data['T1'][batch_idx].to(DEVICE, non_blocking=True)
                    T_amb = chunk_data['T_amb'][batch_idx].to(DEVICE, non_blocking=True)
                    valid_len = chunk_data['valid_len'][batch_idx].to(DEVICE, non_blocking=True)
                    
                    optimizer.zero_grad()
                    
                    # Forward pass
                    T0_init, T1_init = T0_true[:, 0], T1_true[:, 0]
                    T0_pred, T1_pred = model(P0, P1, T0_init, T1_init, T_amb)
                    
                    # Create mask for valid timesteps
                    seq_len = P0.shape[1]
                    mask = torch.arange(seq_len, device=DEVICE).expand(len(valid_len), seq_len) < valid_len.unsqueeze(1)
                    
                    # Calculate MSE (for backprop)
                    mse_0 = mse_criterion(T0_pred, T0_true)
                    mse_1 = mse_criterion(T1_pred, T1_true)
                    masked_mse = ((mse_0 + mse_1) * mask).sum() / (2 * mask.sum())
                    
                    # Calculate MAE (for tracking interpretability)
                    with torch.no_grad():
                        mae_0 = mae_criterion(T0_pred.detach(), T0_true)
                        mae_1 = mae_criterion(T1_pred.detach(), T1_true)
                        masked_mae = ((mae_0 + mae_1) * mask).sum() / (2 * mask.sum())
    
                    # Backpropagate
                    masked_mse.backward()
                    optimizer.step()
                    
                    train_mse_accum += masked_mse.item()
                    train_mae_accum += masked_mae.item()
                    train_steps += 1
                    
                del chunk_data

    
            avg_train_mse = train_mse_accum / train_steps
            train_rmse = avg_train_mse ** 0.5
            train_mae = train_mae_accum / train_steps
            
            # 2. Validation Phase (using pre-cached chunks)
            model.eval()
            val_mse_accum = 0.0
            val_mae_accum = 0.0
            val_steps = 0
            
            with torch.no_grad():
                for chunk_data in val_chunks_data:
                    total_size = chunk_data['P0'].shape[0]
                    
                    for start_idx in range(0, total_size, BATCH_SIZE):
                        end_idx = min(start_idx + BATCH_SIZE, total_size)
                        
                        # Manual Batch Slicing (Sequential for Val)
                        P0 = chunk_data['P0'][start_idx:end_idx].to(DEVICE, non_blocking=True)
                        P1 = chunk_data['P1'][start_idx:end_idx].to(DEVICE, non_blocking=True)
                        T0_true = chunk_data['T0'][start_idx:end_idx].to(DEVICE, non_blocking=True)
                        T1_true = chunk_data['T1'][start_idx:end_idx].to(DEVICE, non_blocking=True)
                        T_amb = chunk_data['T_amb'][start_idx:end_idx].to(DEVICE, non_blocking=True)
                        valid_len = chunk_data['valid_len'][start_idx:end_idx].to(DEVICE, non_blocking=True)
                        
                        T0_init, T1_init = T0_true[:, 0], T1_true[:, 0]
                        
                        T0_pred, T1_pred = model(P0, P1, T0_init, T1_init, T_amb)
                        
                        seq_len = P0.shape[1]
                        mask = torch.arange(seq_len, device=DEVICE).expand(len(valid_len), seq_len) < valid_len.unsqueeze(1)
                        
                        # MSE
                        mse_0 = mse_criterion(T0_pred, T0_true)
                        mse_1 = mse_criterion(T1_pred, T1_true)
                        masked_mse = ((mse_0 + mse_1) * mask).sum() / (2 * mask.sum())
                        
                        # MAE
                        mae_0 = mae_criterion(T0_pred, T0_true)
                        mae_1 = mae_criterion(T1_pred, T1_true)
                        masked_mae = ((mae_0 + mae_1) * mask).sum() / (2 * mask.sum())
                        
                        val_mse_accum += masked_mse.item()
                        val_mae_accum += masked_mae.item()
                        val_steps += 1
                                            
            avg_val_mse = val_mse_accum / val_steps
            val_rmse = avg_val_mse ** 0.5
            val_mae = val_mae_accum / val_steps
            
            epoch_time = time.perf_counter() - epoch_start_time
            
            # Print Progress
            p = {k: v.item() for k, v in model.get_physical_params().items()}
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Time: {epoch_time:.1f}s | Train RMSE: {train_rmse:.4f} °C (MAE: {train_mae:.4f}) | Val RMSE: {val_rmse:.4f} °C (MAE: {val_mae:.4f}) | LR: {current_lr:.6f}")
            print(f"Params: C: {p['C']:.2f} | h_base0: {p['h_base_0']:.2f} | h_act0: {p['h_active_0']:.2f} | T_thr0: {p['T_thresh_0']:.1f} | h_base1: {p['h_base_1']:.2f} | h_act1: {p['h_active_1']:.2f} | T_thr1: {p['T_thresh_1']:.1f}")

            scheduler.step(val_rmse)
            
            # Logging to CSV
            with open(CSV_LOG_PATH, "a") as f:
                f.write(f"{epoch},{train_rmse:.6f},{val_rmse:.6f},{train_mae:.6f},{val_mae:.6f},{current_lr:.6f},{p['C']:.4f},{p['h_base_0']:.4f},{p['h_base_1']:.4f},{p['h_active_0']:.4f},{p['h_active_1']:.4f},{p['T_thresh_0']:.4f},{p['T_thresh_1']:.4f},{p['beta_0']:.4f},{p['beta_1']:.4f},{p['k01']:.4f},{p['k10']:.4f},{p['q0']:.4f},{p['q1']:.4f},{epoch_time:.2f}\n")
    
            # Checkpointing
            is_best = val_rmse < best_val_loss
            if is_best:
                best_val_loss = val_rmse
    
            checkpoint_state = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_loss': best_val_loss,
                'torch_rng': torch.get_rng_state(),
                'numpy_rng': np.random.get_state(),
                'cuda_rng': torch.cuda.get_rng_state() if torch.cuda.is_available() else None
            }
    
            # Save standard resume checkpoint (contains the updated best_val_loss)
            torch.save(checkpoint_state, RESUME_PATH)
    
            # Save best model checkpoint if validation loss improves
            if is_best:
                torch.save(checkpoint_state, BEST_PATH)
                
                with open(SAVE_DIR / "calibrated_physics.json", "w") as f:
                    json.dump(p, f, indent=4)
                print(">>> New best parameters saved! <<<")
    
        # End Summary
        end_time = time.perf_counter()
        formatted_time = str(timedelta(seconds=int(end_time - start_time)))
        
        print("\n" + "=" * 70)
        print("=== CALIBRATION COMPLETE ===")
        print(f"Total Time : {formatted_time}")
        print(f"Best Val RMSE : {best_val_loss:.4f} °C")
        print(f"Results saved to: {str(SAVE_DIR).replace(prefix, '')}")
        print("=" * 70)

    finally:
        if gpu_logger_process is not None:
            gpu_logger_process.terminate()
            gpu_logger_process.wait()
            print("[*] Background GPU logger terminated.")

        if hasattr(sys.stdout, 'terminal'):
            sys.stdout.close()
            sys.stdout = sys.stdout.terminal

if __name__ == "__main__":
    main()
    


[SYSTEM] Live GPU Status:
Fri Apr  3 12:12:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 33%   61C    P2             54W /  200W |    7683MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']:  1


[SYSTEM] Manually locking script to GPU 1.



Enter version prefix (e.g., v1_base) [Press Enter for default 'v1']:  ode_v3


--- STARTING: 07_GPU_ODE_CALIBRATION ---
[*] Version Prefix: ode_v3
[*] Device: cuda:1
[*] Saving to: Capstone/Implementation/models/ode_v3
[*] Started continuous GPU logging to: ode_v3_gpu1_timeseries.csv

[SYSTEM] No checkpoint found. Starting fresh training.
Found 7 train chunks and 1 val chunks.

[SYSTEM] Pre-loading 1 validation chunk(s) into RAM...
[SYSTEM] Validation data cached. (1 chunk(s) ready)

--- Epoch 1/50 ---


...

Time: 344.7s | Train RMSE: 3.6053 °C (MAE: 2.5301) | Val RMSE: 3.6890 °C (MAE: 2.7067) | LR: 0.010000
Params: C: 876.41 | h_base0: 3.74 | h_act0: 7.54 | T_thr0: 70.8 | h_base1: 4.11 | h_act1: 8.02 | T_thr1: 67.0
>>> New best parameters saved! <<<

--- Epoch 47/50 ---


Time: 344.6s | Train RMSE: 3.5964 °C (MAE: 2.5226) | Val RMSE: 3.6800 °C (MAE: 2.6983) | LR: 0.010000
Params: C: 882.66 | h_base0: 3.73 | h_act0: 7.62 | T_thr0: 70.8 | h_base1: 4.11 | h_act1: 8.09 | T_thr1: 67.0
>>> New best parameters saved! <<<

--- Epoch 48/50 ---


Time: 342.6s | Train RMSE: 3.5879 °C (MAE: 2.5153) | Val RMSE: 3.6716 °C (MAE: 2.6905) | LR: 0.010000
Params: C: 888.41 | h_base0: 3.73 | h_act0: 7.70 | T_thr0: 70.8 | h_base1: 4.11 | h_act1: 8.16 | T_thr1: 67.0
>>> New best parameters saved! <<<

--- Epoch 49/50 ---


Time: 353.6s | Train RMSE: 3.5816 °C (MAE: 2.5096) | Val RMSE: 3.6638 °C (MAE: 2.6832) | LR: 0.010000
Params: C: 893.71 | h_base0: 3.73 | h_act0: 7.78 | T_thr0: 70.8 | h_base1: 4.11 | h_act1: 8.24 | T_thr1: 67.0
>>> New best parameters saved! <<<

--- Epoch 50/50 ---


Time: 346.2s | Train RMSE: 3.5738 °C (MAE: 2.5032) | Val RMSE: 3.6565 °C (MAE: 2.6762) | LR: 0.010000
Params: C: 898.62 | h_base0: 3.73 | h_act0: 7.86 | T_thr0: 70.8 | h_base1: 4.10 | h_act1: 8.31 | T_thr1: 67.0
>>> New best parameters saved! <<<

=== CALIBRATION COMPLETE ===
Total Time : 4:47:50
Best Val RMSE : 3.6565 °C
Results saved to: /home/sanke24839/Capstone/Implementation/models/ode_v3
[*] Background GPU logger terminated.


In [2]:
"""
===============================================================================
Script Name: 02_test_evaluation.py
Description: Evaluates the calibrated Physics-Informed ODE on the unseen
             test set. Runs high-speed batched inference, stitches chunks 
             back into full physical HPC jobs using a composite key, 
             computes tail-risk/drift metrics, and generates HTML diagnostics.
===============================================================================
"""

import os
import sys
import time
import warnings
import subprocess
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta

warnings.filterwarnings('ignore')

# --- 1. PHYSICS MODEL (Must match training exactly to load weights) ---
class ThermalODEModel(nn.Module):
    def __init__(self):
        super(ThermalODEModel, self).__init__()
        
        prior_C         = 141.735
        prior_k01       = 0.0780
        prior_k10       = 0.0281
        prior_q         = 0.0
        
        prior_h_base    = 2.0
        prior_h_active  = 8.0
        prior_T_thresh  = 65.0
        prior_beta      = 0.5
        
        self.bounds = {
            'C':          (100.0, 1000.0),
            'k01':        (0.0, 0.5),
            'k10':        (0.0, 0.5),
            'q0':         (-10.0, 10.0),
            'q1':         (-10.0, 10.0),
            'h_base_0':   (0.5, 10.0),
            'h_base_1':   (0.5, 10.0),
            'h_active_0': (0.0, 30.0),
            'h_active_1': (0.0, 30.0),
            'T_thresh_0': (40.0, 85.0),
            'T_thresh_1': (40.0, 85.0),
            'beta_0':     (0.05, 2.0),
            'beta_1':     (0.05, 2.0)
        }
        
        def inv_sigmoid(val, low, high):
            norm = (val - low) / (high - low)
            norm = max(min(norm, 0.999), 0.001)
            return torch.log(torch.tensor(norm / (1.0 - norm)))

        self.raw_C          = nn.Parameter(inv_sigmoid(prior_C, *self.bounds['C']))
        self.raw_k01        = nn.Parameter(inv_sigmoid(prior_k01, *self.bounds['k01']))
        self.raw_k10        = nn.Parameter(inv_sigmoid(prior_k10, *self.bounds['k10']))
        self.raw_q0         = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q0']))
        self.raw_q1         = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q1']))
        
        self.raw_h_base_0   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_0']))
        self.raw_h_active_0 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_0']))
        self.raw_T_thresh_0 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_0']))
        self.raw_beta_0     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_0']))
        
        self.raw_h_base_1   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_1']))
        self.raw_h_active_1 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_1']))
        self.raw_T_thresh_1 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_1']))
        self.raw_beta_1     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_1']))

    def get_physical_params(self):
        p = {}
        for name, bound in self.bounds.items():
            raw_val = getattr(self, f"raw_{name}")
            low, high = bound
            p[name] = low + torch.sigmoid(raw_val) * (high - low)
        return p

    def forward(self, P0, P1, T0_init, T1_init, T_amb):
        params = self.get_physical_params()
        batch_size, seq_len = P0.shape
        DT = 0.11
        
        T0_preds = torch.empty((batch_size, seq_len), device=P0.device)
        T1_preds = torch.empty((batch_size, seq_len), device=P1.device)
        
        T0_curr, T1_curr = T0_init, T1_init
        T0_preds[:, 0] = T0_curr
        T1_preds[:, 0] = T1_curr
        
        inv_C = 1.0 / params['C']
        
        for t in range(seq_len - 1):
            h0_curr = params['h_base_0'] + params['h_active_0'] * torch.sigmoid(params['beta_0'] * (T0_curr - params['T_thresh_0']))
            h1_curr = params['h_base_1'] + params['h_active_1'] * torch.sigmoid(params['beta_1'] * (T1_curr - params['T_thresh_1']))
            
            dT0 = (P0[:, t] + params['k01'] * P1[:, t] - h0_curr * T0_curr + h0_curr * T_amb + params['q0']) * inv_C
            dT1 = (P1[:, t] + params['k10'] * P0[:, t] - h1_curr * T1_curr + h1_curr * T_amb + params['q1']) * inv_C
            
            T0_curr = T0_curr + DT * dT0
            T1_curr = T1_curr + DT * dT1
            
            T0_preds[:, t+1] = T0_curr
            T1_preds[:, t+1] = T1_curr
            
        return T0_preds, T1_preds


# --- 2. GPU SELECTION & INITIALIZATION ---
def select_gpu():
    """Selects GPU via CLI arg, env var, or interactive prompt."""
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            gpu_id = int(sys.argv[i + 1])
            if gpu_id < torch.cuda.device_count():
                print(f"[SYSTEM] GPU {gpu_id} selected via CLI argument.")
                return gpu_id
            else:
                print(f"[!] CLI GPU {gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        gpu_id = int(env_gpu)
        if gpu_id < torch.cuda.device_count():
            print(f"[SYSTEM] GPU {gpu_id} selected via GPU_ID environment variable.")
            return gpu_id
        else:
            print(f"[!] Env GPU_ID={gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    print("\n[SYSTEM] Live GPU Status:")
    try:
        subprocess.run(["nvidia-smi"])
    except FileNotFoundError:
        print("[WARNING] nvidia-smi not found. Cannot display GPU stats.")

    if not torch.cuda.is_available():
        print("[WARNING] No CUDA GPUs detected. Falling back to CPU.")
        return None

    while True:
        gpu_id = input("\n[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']: ").strip()
        if not gpu_id:
            print("[SYSTEM] Defaulting to GPU 0.")
            return 0
        if gpu_id.isdigit():
            gpu_id_int = int(gpu_id)
            if gpu_id_int < torch.cuda.device_count():
                print(f"[SYSTEM] Manually locking script to GPU {gpu_id_int}.")
                return gpu_id_int
            else:
                print(f"[!] GPU {gpu_id_int} not found. Available GPUs: 0-{torch.cuda.device_count()-1}")
                continue
        print("[!] Invalid input. Please enter a valid integer ID.")

# Helper Metrics
def calc_rmse(true, pred):
    return np.sqrt(np.mean((true - pred)**2))

def calc_mae(true, pred):
    return np.mean(np.abs(true - pred))

# Plotly HTML Generation
def generate_interactive_plot(job_list, stitched_data, filename, title_prefix):
    """Generates an HTML file with separated GPU panes and a dropdown for jobs."""
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=("GPU 0 Temperature", "GPU 1 Temperature"),
        horizontal_spacing=0.05
    )

    traces_per_job = 4 # T0_True, T0_Pred, T1_True, T1_Pred
    buttons = []

    for i, run_id in enumerate(job_list):
        job = stitched_data[run_id]
        t = np.arange(len(job['T0_true']))
        visible = (i == 0) # Only the first job is visible by default
        
        # GPU 0 (Col 1)
        fig.add_trace(go.Scatter(x=t, y=job['T0_true'], mode='lines', line=dict(color='blue'), name="True T0", visible=visible), row=1, col=1)
        fig.add_trace(go.Scatter(x=t, y=job['T0_pred'], mode='lines', line=dict(color='red', dash='dot'), name="Pred T0", visible=visible), row=1, col=1)
        
        # GPU 1 (Col 2)
        fig.add_trace(go.Scatter(x=t, y=job['T1_true'], mode='lines', line=dict(color='blue'), name="True T1", visible=visible), row=1, col=2)
        fig.add_trace(go.Scatter(x=t, y=job['T1_pred'], mode='lines', line=dict(color='red', dash='dot'), name="Pred T1", visible=visible), row=1, col=2)

        # Build dropdown toggle logic
        visibility_array = [False] * (len(job_list) * traces_per_job)
        visibility_array[i*traces_per_job : (i+1)*traces_per_job] = [True, True, True, True]
        
        buttons.append(dict(
            label=f"{run_id} (RMSE: {job['rmse']:.2f}°C)",
            method="update",
            args=[{"visible": visibility_array}, 
                  {"title": f"{title_prefix} - Run ID: {run_id}"}]
        ))
        
    fig.update_layout(
        title=f"{title_prefix} - Run ID: {job_list[0]}",
        updatemenus=[dict(active=0, buttons=buttons, x=0.5, y=1.2, xanchor="center", yanchor="top")],
        template="plotly_white",
        hovermode="x unified",
        height=600
    )
    
    # Force Y-axis to match GPU reality limits roughly
    fig.update_yaxes(title_text="Temperature (°C)", range=[20, 100], row=1, col=1)
    fig.update_yaxes(title_text="Temperature (°C)", range=[20, 100], row=1, col=2)
    
    fig.write_html(filename)


# --- 3. MAIN EXECUTION ---
def main():
    global script_name
    try:
        script_path = Path(__file__).resolve()
        script_name = script_path.stem
        project_root = script_path.parent.parent.parent
    except NameError:
        script_name = "02_test_evaluation"
        current_dir = Path.cwd() 
        project_root = current_dir.parent.parent
    
    # GPU Setup
    gpu_id = select_gpu()
    if gpu_id is not None:
        torch.cuda.set_device(gpu_id)
        DEVICE = torch.device(f"cuda:{gpu_id}")
    else:
        DEVICE = torch.device("cpu")

    user_prefix = input("\nEnter version prefix to load (e.g., v1_base) [Press Enter for default 'v1']: ").strip()
    VERSION_PREFIX = user_prefix if user_prefix else "v1"

    # Paths
    # Adjust `data_dir` to where your test tensors actually sit
    data_dir = project_root / "Implementation" / "data" / "mit-supercloud-dataset" / "gpu" / "dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_tensors"
    models_dir = project_root / "Implementation" / "models" / VERSION_PREFIX
    
    BEST_PATH = models_dir / f"{VERSION_PREFIX}_best_model.pt"
    if not BEST_PATH.exists():
        print(f"[!] Could not find best model at {BEST_PATH}. Exiting.")
        return
        
    EVAL_DIR = models_dir / "evaluation"
    EVAL_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 70)
    print(f"--- STARTING: {script_name.upper()} ---")
    print("=" * 70)

    start_time = time.perf_counter()
    
    # 1. Load Model
    model = ThermalODEModel().to(DEVICE)
    checkpoint = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"[*] Model weights loaded from Epoch {checkpoint['epoch']} (Val RMSE: {checkpoint['best_val_loss']:.4f})")

    # 2. Unbatched Inference
    test_chunks = sorted(list(data_dir.glob("test_segments_part*.pt")))
    print(f"[*] Found {len(test_chunks)} test chunks. Starting inference...")
    
    raw_inferences = []

    with torch.no_grad():
        for chunk_idx, chunk_file in enumerate(tqdm(test_chunks, desc="Inferencing Chunks")):
            chunk_data = torch.load(chunk_file, map_location='cpu', weights_only=False)
            total_size = chunk_data['P0'].shape[0]
            
            # Send entire chunk to VRAM at once
            P0 = chunk_data['P0'].to(DEVICE, non_blocking=True)
            P1 = chunk_data['P1'].to(DEVICE, non_blocking=True)
            T0_true = chunk_data['T0'].to(DEVICE, non_blocking=True)
            T1_true = chunk_data['T1'].to(DEVICE, non_blocking=True)
            T_amb = chunk_data['T_amb'].to(DEVICE, non_blocking=True)
            
            T0_init, T1_init = T0_true[:, 0], T1_true[:, 0]
            
            # Single forward pass for the whole chunk
            T0_pred, T1_pred = model(P0, P1, T0_init, T1_init, T_amb)
            
            # Offload back to CPU as numpy arrays
            T0_pred_np = T0_pred.cpu().numpy()
            T1_pred_np = T1_pred.cpu().numpy()
            T0_true_np = T0_true.cpu().numpy()
            T1_true_np = T1_true.cpu().numpy()
            
            for i in range(total_size):
                raw_inferences.append({
                    'job_id': chunk_data['job_ids'][i],
                    'node_id': chunk_data['node_ids'][i],
                    'valid_len': int(chunk_data['valid_len'][i]),
                    'T0_pred': T0_pred_np[i],
                    'T1_pred': T1_pred_np[i],
                    'T0_true': T0_true_np[i],
                    'T1_true': T1_true_np[i]
                })
            
            del chunk_data
            torch.cuda.empty_cache()

    # 3. Trajectory Stitching (The Composite Key)
    print("\n[*] Inference complete. Stitching segments back into physical jobs...")
    df_raw = pd.DataFrame(raw_inferences)
    df_raw['unique_run_id'] = df_raw['job_id'].astype(str) + "_" + df_raw['node_id'].astype(str)
    
    stitched_jobs = {}
    
    # We rely on pandas groupby preserving the sequential order of rows (which matches chronological segment order)
    for run_id, group in tqdm(df_raw.groupby('unique_run_id'), desc="Stitching Trajectories"):
        full_T0_pred, full_T1_pred = [], []
        full_T0_true, full_T1_true = [], []
        
        for _, row in group.iterrows():
            v_len = row['valid_len']
            full_T0_pred.extend(row['T0_pred'][:v_len])
            full_T1_pred.extend(row['T1_pred'][:v_len])
            full_T0_true.extend(row['T0_true'][:v_len])
            full_T1_true.extend(row['T1_true'][:v_len])
            
        t0_true_np = np.array(full_T0_true)
        t0_pred_np = np.array(full_T0_pred)
        t1_true_np = np.array(full_T1_true)
        t1_pred_np = np.array(full_T1_pred)
        
        # Combine GPUs for a single job-level RMSE
        job_true = np.concatenate([t0_true_np, t1_true_np])
        job_pred = np.concatenate([t0_pred_np, t1_pred_np])
        
        stitched_jobs[run_id] = {
            'T0_true': t0_true_np, 'T0_pred': t0_pred_np,
            'T1_true': t1_true_np, 'T1_pred': t1_pred_np,
            'rmse': calc_rmse(job_true, job_pred),
            'length': len(job_true) // 2
        }

    # 4. Global Statistical & Tail-Risk Evaluation
    print("\n" + "=" * 70)
    print("=== FINAL TEST EVALUATION METRICS ===")
    print("=" * 70)
    
    all_true_list = []
    all_pred_list = []
    
    # ADDED TQDM so you see the progress instead of a frozen cell
    for job in tqdm(stitched_jobs.values(), desc="Aggregating Arrays"):
        # Append the numpy arrays directly to the list (Do NOT extend the individual elements)
        all_true_list.extend([job['T0_true'], job['T1_true']])
        all_pred_list.extend([job['T0_pred'], job['T1_pred']])
        
    print("\n[*] Concatenating millions of timesteps (this will take a few seconds)...")
    all_true = np.concatenate(all_true_list)
    all_pred = np.concatenate(all_pred_list)
    all_abs_errors = np.abs(all_true - all_pred)
    
    # Global Baseline
    global_rmse = calc_rmse(all_true, all_pred)
    global_mae = calc_mae(all_true, all_pred)
    
    # Outliers
    p95 = np.percentile(all_abs_errors, 95)
    p99 = np.percentile(all_abs_errors, 99)
    max_err = np.max(all_abs_errors)
    mean_err = np.mean(all_pred - all_true) # Directional bias
    
    # Danger Zone (>70C)
    danger_mask = all_true > 70.0
    if np.any(danger_mask):
        danger_rmse = calc_rmse(all_true[danger_mask], all_pred[danger_mask])
    else:
        danger_rmse = float('nan')

    # Temporal Drift 
    first_10_rmses, last_10_rmses = [], []
    for job in stitched_jobs.values():
        idx_10_pct = max(1, int(job['length'] * 0.10))
        
        t0_true, t1_true = job['T0_true'], job['T1_true']
        t0_pred, t1_pred = job['T0_pred'], job['T1_pred']
        
        first_true = np.concatenate([t0_true[:idx_10_pct], t1_true[:idx_10_pct]])
        first_pred = np.concatenate([t0_pred[:idx_10_pct], t1_pred[:idx_10_pct]])
        last_true = np.concatenate([t0_true[-idx_10_pct:], t1_true[-idx_10_pct:]])
        last_pred = np.concatenate([t0_pred[-idx_10_pct:], t1_pred[-idx_10_pct:]])
        
        first_10_rmses.append(calc_rmse(first_true, first_pred))
        last_10_rmses.append(calc_rmse(last_true, last_pred))

    drift_first = np.mean(first_10_rmses)
    drift_last = np.mean(last_10_rmses)

    print(f"\nTotal Physical Jobs Reconstructed : {len(stitched_jobs)}")
    print(f"Total Timesteps Evaluated         : {len(all_true):,}")
    print("-" * 70)
    print(f"[Baseline] Global RMSE          : {global_rmse:.4f} °C")
    print(f"[Baseline] Global MAE           : {global_mae:.4f} °C")
    print(f"[Baseline] Directional Bias     : {mean_err:+.4f} °C (>0 means model predicts hotter)")
    print("-" * 70)
    print(f"[Tail-Risk] 95th Percentile Err : {p95:.4f} °C")
    print(f"[Tail-Risk] 99th Percentile Err : {p99:.4f} °C")
    print(f"[Tail-Risk] Absolute Max Error  : {max_err:.4f} °C")
    print("-" * 70)
    print(f"[Extremes] RMSE in Danger Zone  : {danger_rmse:.4f} °C (Only evaluated where True > 70°C)")
    print("-" * 70)
    print(f"[Stability] First 10% RMSE      : {drift_first:.4f} °C")
    print(f"[Stability] Last 10% RMSE       : {drift_last:.4f} °C")
    print("=" * 70)

    # 5. Interactive HTML Plotting
    print("\n[*] Generating Interactive Plotly Diagnostics...")
    
    # Sort jobs by RMSE
    sorted_job_ids = sorted(stitched_jobs.keys(), key=lambda k: stitched_jobs[k]['rmse'])
    
    best_10 = sorted_job_ids[:10]
    worst_10 = sorted_job_ids[-10:][::-1] # Reverse so the absolute worst is index 0
    
    mid_idx = len(sorted_job_ids) // 2
    median_10 = sorted_job_ids[mid_idx - 5 : mid_idx + 5]

    generate_interactive_plot(best_10, stitched_jobs, EVAL_DIR / "best_10_jobs.html", "Top 10 Best Predictions")
    generate_interactive_plot(median_10, stitched_jobs, EVAL_DIR / "median_10_jobs.html", "Median 10 Average Predictions")
    generate_interactive_plot(worst_10, stitched_jobs, EVAL_DIR / "worst_10_jobs.html", "Bottom 10 Worst Predictions")
    
    print(f"[SUCCESS] Diagnostic HTML files saved to: {EVAL_DIR}")

    end_time = time.perf_counter()
    formatted_time = str(timedelta(seconds=int(end_time - start_time)))
    
    print("\n" + "=" * 70)
    print("=== EVALUATION COMPLETE ===")
    print(f"Total Time : {formatted_time}")
    print("=" * 70)

if __name__ == "__main__":
    main()


[SYSTEM] Live GPU Status:
Fri Apr  3 18:46:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 36%   64C    P5             64W /  200W |    7870MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']:  1


[SYSTEM] Manually locking script to GPU 1.



Enter version prefix to load (e.g., v1_base) [Press Enter for default 'v1']:  ode_v3


--- STARTING: 02_TEST_EVALUATION ---
[*] Model weights loaded from Epoch 50 (Val RMSE: 3.6565)
[*] Found 1 test chunks. Starting inference...


...


[*] Inference complete. Stitching segments back into physical jobs...


Stitching Trajectories: 100%|██████████| 553/553 [03:15<00:00,  2.83it/s]



=== FINAL TEST EVALUATION METRICS ===


Aggregating Arrays: 100%|██████████| 553/553 [00:00<00:00, 397710.92it/s]



[*] Concatenating millions of timesteps (this will take a few seconds)...

Total Physical Jobs Reconstructed : 553
Total Timesteps Evaluated         : 443,815,380
----------------------------------------------------------------------
[Baseline] Global RMSE          : 3.3187 °C
[Baseline] Global MAE           : 2.4031 °C
[Baseline] Directional Bias     : +0.4249 °C (>0 means model predicts hotter)
----------------------------------------------------------------------
[Tail-Risk] 95th Percentile Err : 6.7916 °C
[Tail-Risk] 99th Percentile Err : 10.3131 °C
[Tail-Risk] Absolute Max Error  : 27.1783 °C
----------------------------------------------------------------------
[Extremes] RMSE in Danger Zone  : 5.9985 °C (Only evaluated where True > 70°C)
----------------------------------------------------------------------
[Stability] First 10% RMSE      : 3.1381 °C
[Stability] Last 10% RMSE       : 2.9284 °C

[*] Generating Interactive Plotly Diagnostics...
[SUCCESS] Diagnostic HTML files sav